# ML Pipeline Quickstart

This notebook exercises the new ML pipeline utilities: S3 helpers, audio decoding, mel feature generation, and PCA.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Add repo/python to path so `ml` is importable
repo_root = Path.cwd().parents[2]
sys.path.insert(0, str(repo_root / 'python'))

from ml.storage import read_parquet_uri, read_audio_path_ffmpeg
from ml.features.audio_mel import MelSegmentsConfig, generate_mel_segments
from ml.analysis.pca import pca_from_mel_df, pca_from_tabular_df


## 1) Decode a WebM with ffmpeg (local)

In [ ]:
sample_webm = Path('/run/media/cbaguilar/T7/wyze_dump/camera=Santa_Teresa_Cam_1/date=2026-02-25/Santa_Teresa_Cam_1_2026-02-25T07-23-07-263Z_chunk=000001.webm')
y, sr, subtype = read_audio_path_ffmpeg(sample_webm, sample_rate=16000, mono=True)
(y.shape, sr, subtype)

## 2) Generate mel features for a small subset
This will write shards + a manifest under the output directory.

In [ ]:
cfg = MelSegmentsConfig(
    segments_root=Path('/run/media/cbaguilar/T7/wyze_dump'),
    segments_manifest='',
    out_dir=Path('/tmp/wyze_mel_test'),
    site='wyze',
    limit_segments=1000,
    sample_rate=16000,
    target_seconds=10.0,
    decoder='ffmpeg',
)
manifest_path = generate_mel_segments(cfg)
manifest_path

## 3) Load manifest and run PCA on mel shards

In [ ]:
mel_df = pd.read_parquet(manifest_path)
pca_out_dir = Path('/tmp/wyze_pca_test')
res = pca_from_mel_df(mel_df, pca_out_dir, limit=100, sample_mode='random')
res

### Plotly scatter with click-to-open

In [ ]:
import plotly.graph_objects as go
import webbrowser
from pathlib import Path

pca_points = pd.read_parquet(res.points_path)
label_col = None
for c in ['segment_path', 'segment_relpath', 'source_name', 'mel_shard_path']:
    if c in pca_points.columns:
        label_col = c
        break

labels = pca_points[label_col].astype(str).tolist() if label_col else [str(i) for i in range(len(pca_points))]

fig = go.FigureWidget(
    data=[go.Scatter(
        x=pca_points['pc1'],
        y=pca_points['pc2'],
        mode='markers',
        text=labels,
        hovertemplate='%{text}<br>pc1=%{x:.3f}<br>pc2=%{y:.3f}<extra></extra>',
        marker=dict(size=6, opacity=0.8)
    )],
    layout=go.Layout(title='PCA on mel shards (clickable)', height=500)
)

def handle_click(trace, points, state):
    if not points.point_inds:
        return
    idx = points.point_inds[0]
    path = labels[idx]
    print('Clicked:', path)
    if path.startswith('/') and Path(path).exists():
        webbrowser.open(f'file://{path}')

fig.data[0].on_click(handle_click)
fig


## 4) PCA on tabular data (example)

In [5]:
tab_df = pd.DataFrame({
    'a': [1,2,3,4,5],
    'b': [2,1,2,1,2],
    'c': [5,4,3,2,1],
})
tab_res = pca_from_tabular_df(tab_df, Path('/tmp/tab_pca'))
tab_res

PCAResult(points_path=PosixPath('/tmp/tab_pca/pca_points.parquet'), model_path=PosixPath('/tmp/tab_pca/pca_model.npz'), meta_path=PosixPath('/tmp/tab_pca/pca_meta.json'))